# UQRoute-TC — Qwen 7B repeated-sample development subset

**Scope:** Ten seeded generations on each of the same 30 selected development cases across five benchmarks. Each selected base task contributes a clean case and one perturbation. The benchmark split, selection, and model revision remain pinned; held-out groups are excluded.

**Recorded evidence:** A run manifest, per-request responses and seeds, token evidence, parser outcomes, timing, and status-aware canonical and exact-string clusters. Saved terminal outcomes retain their original response. The audit includes truncated or failed slots in the ten-sample denominator.


## 1. Restore pinned inputs


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/UQRoute-TC')
root.mkdir(parents=True, exist_ok=True)
benchmark = Path('/content/robustbench-tc-release')
benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
if not benchmark.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WillChow66/robustbench-tc-release.git', str(benchmark)], check=True)
subprocess.run(['git', '-C', str(benchmark), 'checkout', '--detach', benchmark_revision], check=True)
assert subprocess.check_output(['git', '-C', str(benchmark), 'rev-parse', 'HEAD'], text=True).strip() == benchmark_revision
data_dir = benchmark / 'hf_data/datasets/api_eval'
assert not subprocess.check_output(['git', '-C', str(benchmark), 'status', '--porcelain', '--', str(data_dir)], text=True).strip(), 'Benchmark data has local changes'

project = Path('/content/uqroute-tc-repeated-check')
code_revision = '7c178b4c038e6af44a5c4a6599a5d3bf390e0bbf'
if not project.exists():
    subprocess.run(['git', 'clone', 'https://github.com/RavindraSSK/uqroute-tc.git', str(project)], check=True)
subprocess.run(['git', '-C', str(project), 'fetch', 'origin', 'feat/task2-canonical-parser'], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', '--detach', code_revision], check=True)
assert subprocess.check_output(['git', '-C', str(project), 'rev-parse', 'HEAD'], text=True).strip() == code_revision
print('Benchmark:', benchmark_revision)
print('Project code:', code_revision)


## 2. Serve Qwen 7B on the L4


In [ ]:
import importlib.metadata
import time
import urllib.request

try:
    installed_vllm = importlib.metadata.version('vllm')
except importlib.metadata.PackageNotFoundError:
    installed_vllm = None
packages = ['openai>=1.50,<3', '-e', str(project)]
if installed_vllm != '0.30.0':
    packages.insert(0, 'vllm==0.30.0')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], check=True)
import torch
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0), 'Expected an L4 GPU'
assert importlib.metadata.version('vllm') == '0.30.0'

model_id = 'Qwen/Qwen2.5-7B-Instruct'
model_revision = 'a09a35458c702b33eeacc393d103063234e8bc28'
endpoint = 'http://127.0.0.1:8000/v1'
server_log = Path('/content/uqroute_repeated_qwen7b_server.log')
def served_models():
    try:
        with urllib.request.urlopen(f'{endpoint}/models', timeout=3) as response:
            return [item['id'] for item in json.load(response)['data']]
    except Exception:
        return None

active = served_models()
if active is not None and model_id not in active:
    raise RuntimeError(f'Port 8000 is serving another model: {active}')
if active and model_id in active:
    processes = subprocess.check_output(['ps', '-eo', 'args='], text=True)
    pinned_command = f'vllm serve {model_id} --revision {model_revision}'
    assert pinned_command in processes, 'Existing Qwen 7B server revision could not be verified'
    print('Server ready:', model_id)
else:
    processes = subprocess.check_output(['ps', '-eo', 'args='], text=True)
    if f'vllm serve {model_id}' in processes:
        server = None
        print('Existing Qwen 7B server is starting.')
    else:
        with server_log.open('w') as log:
            server = subprocess.Popen([
                'vllm', 'serve', model_id, '--revision', model_revision,
                '--tokenizer-revision', model_revision, '--dtype', 'bfloat16',
                '--max-model-len', '8192', '--gpu-memory-utilization', '0.88',
                '--port', '8000'], stdout=log, stderr=subprocess.STDOUT,
                start_new_session=True)
        print('Started server PID:', server.pid)
    for attempt in range(120):
        active = served_models()
        if active and model_id in active:
            print('Server ready:', model_id)
            break
        if server is not None and server.poll() is not None:
            print('\n'.join(server_log.read_text(errors='replace').splitlines()[-35:]))
            raise RuntimeError('Server exited during startup')
        time.sleep(5)
    else:
        print('\n'.join(server_log.read_text(errors='replace').splitlines()[-35:]))
        raise TimeoutError('Server was not ready within ten minutes')
print('GPU:', torch.cuda.get_device_name(0), '| vLLM:', importlib.metadata.version('vllm'))


## 3. Generate and save the selected development subset


In [ ]:
from collections import Counter
from openai import OpenAI
sys.path.insert(0, str(benchmark / 'scripts'))
sys.path.insert(0, str(project / 'src'))
from run_eval import build_messages, load_samples, parse_tool_calls
from uqroute_tc.data.repeated_subset import select_repeated_subset
from uqroute_tc.inference.pilot import _atomic_json, token_evidence
from uqroute_tc.parsing.canonical import parse_prediction

split_path = project / 'docs/splits/group_split_v1.json'
split = json.loads(split_path.read_text())
assert split['benchmark_revision'] == benchmark_revision
split_sha256 = hashlib.sha256(split_path.read_bytes()).hexdigest()
assert split_sha256 == '688c1013d8fe9d5c1b454d14d4cbe215659e48adc7bdfd63175580e83e2eb0db'
selected = select_repeated_subset(data_dir, split)
selection_digest = hashlib.sha256(json.dumps(selected, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
assert selection_digest == '082e150679277d7d55eaa88b0b82cb236a82a32a0d6853c2cae8dc307b41fa67'
assert len(selected) == 30
by_file = {}
for source_file in {row['source_file'] for row in selected}:
    rows = load_samples(data_dir / source_file)
    indexed = {str(row['id']): row for row in rows}
    assert len(indexed) == len(rows), f'Duplicate IDs in {source_file}'
    by_file[source_file] = indexed
for item in selected:
    sample = by_file[item['source_file']][item['case_id']]
    assert sample['benchmark'] == item['benchmark']
    assert split['assignments'][item['base_id']] == {
        'benchmark': item['benchmark'], 'population': 'primary', 'split': 'development'}

output_dir = root / 'pilots/qwen25-7b-repeated-dev30-v1'
manifest = {
    'benchmark_revision': benchmark_revision, 'split_sha256': split_sha256,
    'selection_digest': selection_digest, 'selected_cases': selected,
    'code_revision': code_revision, 'model_id': model_id,
    'model_revision': model_revision, 'tokenizer_revision': model_revision,
    'engine_version': '0.30.0', 'dtype': 'bfloat16', 'max_model_len': 8192,
    'endpoint': endpoint, 'mode': 'repeated_sampling_development_subset',
    'temperature': 0.7, 'top_p': 1.0, 'max_tokens': 1024,
    'logprobs': True, 'top_logprobs': 5,
    'sample_seeds': list(range(1729, 1739)), 'request_n': 1,
}
manifest_path = output_dir / 'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Saved run has different settings'
else:
    assert not output_dir.exists() or not any(output_dir.iterdir()), 'Existing output has no manifest'
    _atomic_json(manifest_path, manifest)
assert model_id in (served_models() or []), 'Qwen 7B server unavailable'
client = OpenAI(base_url=endpoint, api_key='EMPTY', timeout=120, max_retries=0)

def record_path(item, index):
    key = item['source_file'] + '\0' + item['case_id']
    folder = hashlib.sha256(key.encode()).hexdigest()
    return output_dir / 'records' / folder / f'sample_{index:02d}.json'

processed = resumed = failed = 0
for case_number, item in enumerate(selected, 1):
    sample = by_file[item['source_file']][item['case_id']]
    messages = build_messages(sample)
    case_status = Counter()
    for index, seed in enumerate(manifest['sample_seeds']):
        path = record_path(item, index)
        previous_attempts = []
        if path.exists():
            old = json.loads(path.read_text())
            assert (old['case_id'], old['source_file'], old['sample_index'], old['seed']) == (
                item['case_id'], item['source_file'], index, seed)
            if old['status'] in {'complete', 'invalid_evidence', 'truncated'}:
                resumed += 1
                case_status[old['status']] += 1
                continue
            previous_attempts = old.get('previous_attempts', []) + [
                {key: value for key, value in old.items() if key != 'previous_attempts'}]
        started = time.perf_counter()
        try:
            response = client.chat.completions.create(
                model=model_id, messages=messages,
                temperature=manifest['temperature'], top_p=manifest['top_p'],
                max_tokens=manifest['max_tokens'], logprobs=True,
                top_logprobs=manifest['top_logprobs'], seed=seed, n=1)
            response_data = response.model_dump(mode='json')
            assert len(response_data['choices']) == 1
            answer = response_data['choices'][0]
            raw = answer['message'].get('content') or ''
            evidence = token_evidence(answer)
            parsed = parse_prediction(raw, sample['benchmark'], parse_tool_calls)
            status = 'truncated' if answer.get('finish_reason') == 'length' else (
                'complete' if evidence['valid'] else 'invalid_evidence')
            error = None if status == 'complete' else (
                'generation reached max_tokens' if status == 'truncated' else evidence['reason'])
            calls = list(parsed.calls)
            finish_reason = answer.get('finish_reason')
            usage = response_data.get('usage')
        except Exception as exc:
            response_data = None
            raw, calls, finish_reason, usage = '', [], None, None
            evidence, parsed = {'valid': False, 'reason': 'request error'}, None
            status, error = 'request_error', f'{type(exc).__name__}: {exc}'
        _atomic_json(path, {
            'case_id': item['case_id'], 'source_file': item['source_file'],
            'benchmark': sample['benchmark'], 'sample_index': index, 'seed': seed,
            'status': status, 'error': error, 'previous_attempts': previous_attempts,
            'elapsed_seconds': time.perf_counter() - started,
            'finish_reason': finish_reason, 'usage': usage,
            'raw_output': raw, 'parsed_calls': calls,
            'parser_status': parsed.status if parsed else 'request_error',
            'token_evidence': evidence, 'response': response_data,
        })
        processed += 1
        failed += status != 'complete'
        case_status[status] += 1
    print(f'{case_number}/30 | {item["benchmark"]} | {item["source_file"]} | '
          f'{dict(case_status)}', flush=True)
print(f'Requests: processed={processed} resumed={resumed} failed={failed}')
print('Records:', output_dir)


## 4. Audit saved outcomes and status-aware clusters


In [ ]:
from collections import Counter
sys.path.insert(0, str(benchmark / 'scripts'))
sys.path.insert(0, str(project / 'src'))
from run_eval import parse_tool_calls
from uqroute_tc.inference.pilot import _atomic_json, token_evidence
from uqroute_tc.parsing.canonical import parse_prediction
from uqroute_tc.uncertainty.repeated import cluster_scores
from google.colab import files

def record_path(item, index):
    key = item['source_file'] + '\0' + item['case_id']
    folder = hashlib.sha256(key.encode()).hexdigest()
    return output_dir / 'records' / folder / f'sample_{index:02d}.json'

def status_clusters(rows, benchmark_name):
    canonical = []
    exact = []
    outcomes = []
    for row in rows:
        status, raw = row['status'], row['raw_output']
        if status in {'truncated', 'request_error'}:
            canonical.append(json.dumps([status]))
            exact.append(json.dumps([status, raw], ensure_ascii=False))
            outcomes.append(status)
        else:
            outcome = parse_prediction(raw, benchmark_name, parse_tool_calls)
            canonical.append(outcome.key)
            exact.append(json.dumps(['output', raw], ensure_ascii=False))
            outcomes.append(outcome.status)
    return cluster_scores(canonical), cluster_scores(exact), outcomes

entries = []
for item in selected:
    rows = []
    errors = []
    for index, seed in enumerate(manifest['sample_seeds']):
        path = record_path(item, index)
        if not path.exists():
            errors.append({'sample_index': index, 'reason': 'missing saved record'})
            continue
        row = json.loads(path.read_text())
        rows.append(row)
        if (row['case_id'], row['source_file'], row['seed'], row['sample_index']) != (
                item['case_id'], item['source_file'], seed, index):
            errors.append({'sample_index': index, 'reason': 'record identity differs'})
            continue
        status = row['status']
        if status not in {'complete', 'invalid_evidence', 'truncated', 'request_error'}:
            errors.append({'sample_index': index, 'reason': 'unknown generation status'})
            continue
        if status == 'request_error':
            continue
        response = row.get('response') or {}
        choices = response.get('choices') or []
        if len(choices) != 1:
            errors.append({'sample_index': index, 'reason': 'missing saved response choice'})
            continue
        choice = choices[0]
        if (choice.get('message') or {}).get('content', '') != row['raw_output']:
            errors.append({'sample_index': index, 'reason': 'visible response differs'})
            continue
        if status == 'truncated':
            if choice.get('finish_reason') != 'length':
                errors.append({'sample_index': index, 'reason': 'truncation lacks length finish reason'})
            continue
        if status == 'complete' and not token_evidence(choice)['valid']:
            errors.append({'sample_index': index, 'reason': 'chosen-token evidence invalid'})
            continue
        reparsed = parse_tool_calls(row['raw_output'], item['benchmark'])
        if json.loads(json.dumps(reparsed, ensure_ascii=False)) != row['parsed_calls']:
            errors.append({'sample_index': index, 'reason': 'saved parser calls differ after JSON round-trip'})
    status_counts = Counter(row['status'] for row in rows)
    parser_counts = Counter(row['parser_status'] for row in rows)
    entry = {**item, 'record_status_counts': dict(status_counts),
             'parser_status_counts': dict(parser_counts),
             'validation_errors': errors, 'scored': len(rows) == 10 and not errors}
    if entry['scored']:
        canonical, exact, outcomes = status_clusters(rows, item['benchmark'])
        entry.update(canonical_entropy_nats=canonical.entropy_nats,
                     canonical_disagreement=canonical.disagreement,
                     canonical_cluster_sizes=canonical.cluster_sizes,
                     exact_string_entropy_nats=exact.entropy_nats,
                     exact_string_disagreement=exact.disagreement,
                     outcome_statuses=outcomes,
                     elapsed_seconds_sum=sum(row['elapsed_seconds'] for row in rows))
    entries.append(entry)

report = {'manifest': manifest, 'cases': len(entries),
          'scored_cases': sum(entry['scored'] for entry in entries),
          'request_status_counts': dict(sum((Counter(entry['record_status_counts'])
                                              for entry in entries), Counter())),
          'validation_error_cases': sum(bool(entry['validation_errors']) for entry in entries),
          'entries': entries}
report_path = output_dir / 'repeated_dev30_status_audit.json'
_atomic_json(report_path, report)
print('Scored cases:', report['scored_cases'], '/ 30')
print('Request statuses:', report['request_status_counts'])
print('Validation error cases:', report['validation_error_cases'])
for entry in entries:
    if entry['record_status_counts'].get('truncated') or entry['validation_errors']:
        print(entry['benchmark'], entry['source_file'], entry['record_status_counts'],
              'clusters:', entry.get('canonical_cluster_sizes'),
              'errors:', entry['validation_errors'])
print('Audit:', report_path)
files.download(str(report_path))
